In [36]:
import pandas as pd, torch
from PIL import Image
import matplotlib.pyplot as plt
import torch
import cv2
import torchvision.transforms as transforms
from torch.utils.data import Dataset,TensorDataset
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import numpy as np
import os 
from sklearn.model_selection import train_test_split 
import torch.nn as nn  
import matplotlib.pyplot as plt
import warnings
from collections import Counter



data = pd.read_csv("kartdatasett_1169/kartdatasett/etiketter.csv")



dictionary = {"Skog":"Skog","Myr / våtmark":"Myr_vatmark","Kyst / hav":"Kyst_hav","Fjell / snaufjell":"Fjell_snaufjell","Spredt bebyggelse":"Spredt_bebyggelse","Innsjø / elv":"Innsjo_elv","Tett bebyggelse":"Tett_bebyggelse","Jordbruk":"Jordbruk","Bre / snø":"Bre_sno"}
dictionary_nums = {"Skog":0,"Myr / våtmark":1,"Kyst / hav":2,"Fjell / snaufjell":3,"Spredt bebyggelse":4,"Innsjø / elv":5,"Tett bebyggelse":6,"Jordbruk":7,"Bre / snø":8}
dictionary_class_to_nums = {"Skog":0,"Myr_vatmark":1,"Kyst_hav":2,"Fjell_snaufjell":3,"Spredt_bebyggelse":4,"Innsjo_elv":5,"Tett_bebyggelse":6,"Jordbruk":7,"Bre_sno":8}
stier = []
klassenummere = []
for i,row in data.iterrows():
    current_biome = dictionary[row["klasse"]]
    current_img = row["fil"]
    img_link = os.path.join("kartdatasett_1169","kartdatasett",current_biome,current_img)
    if os.path.exists(img_link):
        stier.append(img_link)
        klassenummer = dictionary_nums[row["klasse"]]
        klassenummere.append(klassenummer)

data["stier"] = stier
data["klassenummere"] = klassenummere
cut_data = data[["Skog","Myr / våtmark","Kyst / hav","Fjell / snaufjell","Spredt bebyggelse","Innsjø / elv","Tett bebyggelse","Jordbruk","Bre / snø"]]
sum_each_row = cut_data.sum(axis=1)
row_truth = cut_data.idxmax(axis="columns")
row_other_truth = data["klasse"]
truth_rows = row_truth==row_other_truth
values = row_other_truth.value_counts()

#sample = data.groupby("klasse").sample(n=3,random_state=1)
#fig, axs = plt.subplots(9, 3)
#flat_axs = axs.flat
#fig.set_figwidth(25)
#fig.set_figheight(25)
##for (i,row),ax in zip(sample.iterrows(),flat_axs):
##    image = cv2.imread(row["stier"])
##
##    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
##    ax.set_axis_off()
##    ax.set_title(row["klasse"])
##    ax.imshow(image)
    

##plt.show()

#list_of_image_values = []
#for i, row in data.iterrows():
#    image = cv2.imread(row["stier"])
#    list_of_image_values.append(image.shape)
#
#print(Counter(list_of_image_values))



X_train, X_test, y_train, y_test = train_test_split(
    data["stier"], data["klassenummere"],stratify=data["klassenummere"], test_size=0.2, random_state=42)


def convert_to_stack(input_data):
    return_images = []
    for row in input_data:
        image = cv2.imread(row)
    
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        small_image = cv2.resize(image,(128,128),interpolation=cv2.INTER_AREA)
        return_images.append(small_image)
    train_images = np.stack(return_images)
    return train_images


X_train_images = convert_to_stack(X_train)
X_test_images = convert_to_stack(X_test)

X_train_tensor = torch.from_numpy(X_train_images)
X_test_tensor = torch.from_numpy(X_test_images)

X_train_tensor_permuted = torch.permute(X_train_tensor,(0, 3, 1, 2))
X_test_tensor_permuted = torch.permute(X_test_tensor,(0, 3, 1, 2))

X_train_tensor_float = X_train_tensor_permuted.float()
X_test_tensor_float = X_test_tensor_permuted.float()

X_train_normalized = X_train_tensor_float/255
X_test_normalized = X_test_tensor_float/255

numpy_y_train = y_train.to_numpy()
numpy_y_test = y_test.to_numpy()

tensor_y_train = torch.tensor(numpy_y_train,dtype=torch.long)
tensor_y_test = torch.tensor(numpy_y_test,dtype=torch.long)

#print(tensor_y_train)
#print(tensor_y_test)

dataset_training = TensorDataset(X_train_normalized, tensor_y_train)
dataset_test = TensorDataset(X_test_normalized, tensor_y_test)

train_loader = DataLoader(dataset_training,batch_size=16)
test_loader = DataLoader(dataset_test,batch_size=16)

#print(tensor_y_train)
#print(dataset_training[0])
#print(dataset_training[0][0].shape)

#print(X_train_normalized.dtype)
#print(X_test_normalized.dtype)

#print(torch.max(X_train_normalized))
#print(torch.max(X_test_normalized))

#print(torch.min(X_train_normalized))
#print(torch.min(X_test_normalized))


#print(X_train_images.shape)
#print(X_test_images.shape)

#test_image = X_train_images[200]
#plt.imshow(test_image)
#plt.show()


#sum_of_differences = 0
#for row1, row2 in zip(X_test,y_test):
#    gate = os.path.dirname(row1)
#    klasse = os.path.basename(gate)
#    klasse_nr = dictionary_class_to_nums[klasse]
#    if klasse_nr != row2:
#        sum_of_differences += 1
#print(sum_of_differences)    

#print(X_train.index.equals(y_train.index))

#X_train_set = set(X_train)
#X_test_set = set(X_test)

#print(X_train_set.intersection(X_test_set))

#print(X_train.shape)
#print(X_test.shape)

#print(y_train.value_counts())
#print(y_test.value_counts())




tensor([3, 5, 3, 5, 5, 1, 3, 0, 3, 3, 5, 3, 3, 2, 1, 2, 3, 7, 0, 3, 4, 3, 5, 1,
        0, 3, 3, 3, 3, 7, 5, 5, 5, 1, 3, 7, 7, 0, 5, 5, 7, 8, 0, 1, 1, 3, 6, 6,
        3, 0, 1, 3, 0, 1, 3, 3, 5, 1, 0, 5, 5, 3, 5, 5, 0, 5, 5, 4, 6, 3, 1, 5,
        3, 0, 1, 1, 3, 3, 5, 3, 3, 3, 5, 1, 3, 7, 0, 4, 5, 3, 2, 3, 1, 3, 8, 7,
        8, 3, 3, 6, 5, 6, 3, 3, 6, 2, 1, 5, 5, 1, 4, 4, 3, 3, 4, 3, 2, 2, 1, 1,
        1, 3, 4, 1, 5, 3, 5, 3, 0, 3, 5, 3, 0, 3, 3, 5, 7, 3, 3, 5, 1, 0, 1, 3,
        3, 2, 0, 1, 0, 3, 2, 3, 0, 4, 2, 1, 4, 8, 5, 3, 3, 5, 5, 2, 3, 6, 8, 5,
        2, 5, 3, 4, 3, 3, 3, 8, 7, 3, 2, 1, 5, 3, 3, 0, 7, 3, 5, 0, 5, 3, 6, 4,
        3, 3, 8, 5, 5, 3, 3, 1, 5, 6, 4, 6, 4, 5, 3, 6, 0, 8, 4, 1, 0, 1, 5, 6,
        6, 1, 0, 4, 8, 5, 3, 1, 3, 6, 5, 1, 1, 5, 1, 7, 5, 7, 5, 5, 5, 6, 1, 1,
        1, 0, 1, 6, 3, 2, 5, 6, 5, 5, 2, 7, 3, 2, 3, 1, 0, 4, 1, 5, 3, 3, 3, 5,
        3, 1, 5, 7, 7, 8, 4, 7, 5, 5, 3, 1, 1, 1, 1, 4, 0, 7, 7, 1, 5, 1, 5, 3,
        3, 1, 0, 3, 1, 7, 5, 3, 5, 0, 3,